# 📊 Metadata Experiments (YouTube Takeout CSV)
Explore and clean a YouTube Takeout CSV, extract `artist/title`, deduplicate, and prepare for enrichment.

**Requirements:** `pip install pandas numpy python-slugify matplotlib`

**Inputs:** Path to a `.csv` export.
**Outputs:** Cleaned/enriched CSV to `outputs/metadata/cleaned.csv`. Enrichment stubs included.

In [ ]:
# !pip install pandas numpy python-slugify matplotlib


In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from slugify import slugify

CSV_PATH = 'path/to/Takeout.csv'  # <-- change me
OUT_DIR = Path('outputs/metadata')
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
print('Columns:', list(df.columns))
df.head()

## 🧹 Normalize Title/Artist Columns
We try common column names, then parse `title - artist` patterns as fallback.

In [ ]:
def pick(df, names):
    for n in names:
        if n in df.columns: return n
    return None

title_key = pick(df, ['Title','title','Video Title','Song','Track'])
artist_key = pick(df, ['Artist','artist','Channel','channel','Creator','creator'])

def split_title_artist(text):
    if not isinstance(text, str): return None, None
    # common patterns: "Artist - Title" or "Title - Artist"
    parts = re.split(r'\s+-\s+', text)
    if len(parts) == 2:
        # heuristic: artist names are shorter tokens sometimes; keep both variants
        return parts[0], parts[1]
    return None, None

df['_title'] = df[title_key] if title_key else ''
if not artist_key:
    a_guess, t_guess = [], []
    for val in df['_title']:
        a,t = split_title_artist(val)
        a_guess.append(a)
        t_guess.append(t)
    df['_artist'] = a_guess
    df['_track'] = t_guess
else:
    df['_artist'] = df[artist_key]
    df['_track'] = df[title_key] if title_key else ''

df['_artist'] = df['_artist'].fillna('Unknown').astype(str).str.strip()
df['_track']  = df['_track'].fillna('Unknown').astype(str).str.strip()
df['_slug']   = [slugify(f"{a}-{t}") for a,t in zip(df['_artist'], df['_track'])]

print(df[['_artist','_track','_slug']].head())

## 🧪 Simple Stats

In [ ]:
top_artists = df['_artist'].value_counts().head(20)
top_artists.plot(kind='bar', figsize=(10,4))
plt.title('Top Artists')
plt.tight_layout()
plt.show()

print(top_artists)

## 🔗 Enrichment Stubs
Below are functions you can later wire to real services (MusicBrainz/Spotify/YouTube Data API).
For now, they return dummy values.

In [ ]:
from typing import Dict
def enrich_track(artist: str, title: str) -> Dict:
    # TODO: call MusicBrainz/Spotify/YouTube for real metadata
    return {
        'artist_canonical': artist.title(),
        'title_canonical': title.title(),
        'album': None,
        'year': None,
        'isrc': None,
        'confidence': 0.5,
    }

enriched = []
for _,row in df.iterrows():
    enriched.append(enrich_track(row['_artist'], row['_track']))
enriched_df = pd.DataFrame(enriched)
out = pd.concat([df, enriched_df], axis=1)
out_path = OUT_DIR / 'cleaned.csv'
out.to_csv(out_path, index=False)
print('Saved', out_path)